In [ ]:
# §0 — Install & imports
# !pip install torch torchvision jupyter numpy matplotlib pandas scikit-learn umap-learn tqdm

import os, sys, random, time, pickle, warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score
from tqdm import tqdm
import umap
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

CONFIG = {
    'num_points':      1024,
    'num_classes':     40,
    'batch_size':      32,
    'lr':              1e-3,
    'lr_step':         20,
    'lr_gamma':        0.5,
    'weight_decay':    1e-4,
    'feat_reg_weight': 0.001,
    'jitter_sigma':    0.01,
    'jitter_clip':     0.05,
    'seed':            42,
    'num_workers':     0,
    'device':          'cuda'  if torch.cuda.is_available()          else
                       'mps'   if torch.backends.mps.is_available()  else 'cpu',
}
DEVICE = torch.device(CONFIG['device'])
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
elif DEVICE.type == 'mps':
    print("  Apple Silicon GPU (MPS)")

In [ ]:
# §0 — Local paths
BASE_DIR    = Path('~/Desktop/dl-project').expanduser()
DATA_DIR    = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
CKPT_DIR    = BASE_DIR / 'checkpoints'
for d in [DATA_DIR, RESULTS_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Base dir:", BASE_DIR)

In [ ]:
# §1 — Locate ModelNet40 dataset
# Expected structure: data/.../ModelNet40/<class>/train/*.off
#                                                 <class>/test/*.off
import glob

def find_dataset_dir(root):
    # Look for a folder that contains airplane/train/*.off
    matches = glob.glob(str(root) + '/**/airplane/train', recursive=True)
    if matches:
        return Path(matches[0]).parent.parent   # ModelNet40 dir
    return None

DATASET_DIR = find_dataset_dir(DATA_DIR)
assert DATASET_DIR is not None, (
    f"ModelNet40 not found under {DATA_DIR}\n"
    "Expected structure: data/.../ModelNet40/<class>/train/*.off"
)
print(f"Dataset: {DATASET_DIR}")

classes   = sorted([d.name for d in DATASET_DIR.iterdir() if d.is_dir()])
CLASS2IDX = {c: i for i, c in enumerate(classes)}
IDX2CLASS  = {i: c for c, i in CLASS2IDX.items()}
print(f"{len(classes)} classes found")

In [ ]:
# §1 — Dataset class  (reads raw .off mesh files, samples points from surface)
def read_off(path):
    """Parse an OFF file → (vertices, faces) as numpy arrays."""
    with open(path) as f:
        header = f.readline().strip()
        if header == 'OFF':
            counts = f.readline().strip().split()
        else:                          # 'OFF 90714 104773 0' on one line
            counts = header[3:].strip().split()
        n_v, n_f = int(counts[0]), int(counts[1])
        verts = np.array([f.readline().split()[:3] for _ in range(n_v)], dtype=np.float32)
        faces = np.array([f.readline().split()[1:4] for _ in range(n_f)], dtype=np.int32)
    return verts, faces

def sample_mesh(verts, faces, n_pts):
    """Uniformly sample n_pts points from a triangular mesh surface."""
    v0, v1, v2 = verts[faces[:,0]], verts[faces[:,1]], verts[faces[:,2]]
    cross  = np.cross(v1 - v0, v2 - v0)
    areas  = np.maximum(np.nan_to_num(np.sqrt((cross**2).sum(1)) / 2), 1e-10)
    probs  = areas / areas.sum()
    idx    = np.random.choice(len(faces), n_pts, p=probs)
    r1     = np.random.rand(n_pts, 1)
    r2     = np.random.rand(n_pts, 1)
    bad    = (r1 + r2) > 1
    r1[bad], r2[bad] = 1 - r1[bad], 1 - r2[bad]
    return (verts[faces[idx,0]] + r1*(verts[faces[idx,1]]-verts[faces[idx,0]])
                                + r2*(verts[faces[idx,2]]-verts[faces[idx,0]])).astype(np.float32)


class PointCloudDataset(Dataset):
    def __init__(self, root, split='train', num_points=1024, augment=True, classes=None):
        self.root       = Path(root)
        self.split      = split
        self.num_points = num_points
        self.augment    = augment and (split == 'train')
        self.classes    = classes or sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        self.class2idx  = {c: i for i, c in enumerate(self.classes)}

        self.samples = []
        for cls in self.classes:
            folder = self.root / cls / split
            if not folder.exists():
                continue
            for f in sorted(folder.glob('*.off')):
                self.samples.append((f, self.class2idx[cls]))

    def _normalize(self, pts):
        pts = pts - pts.mean(axis=0)
        return pts / (np.max(np.linalg.norm(pts, axis=1)) + 1e-8)

    def _augment(self, pts):
        theta = np.random.uniform(0, 2 * np.pi)
        c, s  = np.cos(theta), np.sin(theta)
        R     = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]], dtype=np.float32)
        pts   = pts @ R.T
        jitter = np.clip(np.random.normal(0, CONFIG['jitter_sigma'], pts.shape),
                         -CONFIG['jitter_clip'], CONFIG['jitter_clip'])
        return (pts + jitter).astype(np.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        verts, faces = read_off(path)
        pts = sample_mesh(verts, faces, self.num_points)
        pts = self._normalize(pts)
        if self.augment:
            pts = self._augment(pts)
        return torch.tensor(pts, dtype=torch.float32), label

In [ ]:
# §1 — DataLoaders
train_ds = PointCloudDataset(DATASET_DIR, 'train', CONFIG['num_points'], augment=True,  classes=classes)
test_ds  = PointCloudDataset(DATASET_DIR, 'test',  CONFIG['num_points'], augment=False, classes=classes)

train_loader = DataLoader(train_ds, CONFIG['batch_size'], shuffle=True,
                          num_workers=CONFIG['num_workers'], drop_last=True)
test_loader  = DataLoader(test_ds,  CONFIG['batch_size'], shuffle=False,
                          num_workers=CONFIG['num_workers'])
print(f"Train: {len(train_ds):,}  Test: {len(test_ds):,}")
print(f"Train batches: {len(train_loader)}  Test batches: {len(test_loader)}")


In [ ]:
# §1 — EDA: class distribution
train_labels = np.array([lbl for _, lbl in train_ds.samples])
test_labels  = np.array([lbl for _, lbl in test_ds.samples])
tr_cnt = np.bincount(train_labels, minlength=40)
te_cnt = np.bincount(test_labels,  minlength=40)
idx    = np.argsort(tr_cnt)[::-1]

fig, ax = plt.subplots(figsize=(18, 5))
x = np.arange(40)
ax.bar(x - .2, tr_cnt[idx], .4, label='Train', color='steelblue', alpha=.85)
ax.bar(x + .2, te_cnt[idx], .4, label='Test',  color='coral',     alpha=.85)
ax.set_xticks(x)
ax.set_xticklabels([IDX2CLASS[i] for i in idx], rotation=90, fontsize=8)
ax.set_ylabel('Samples'); ax.set_title('ModelNet40 — Class Distribution')
ax.legend(); plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig1a_class_distribution.png', dpi=150); plt.show()


In [ ]:
# §1 — EDA: 3D sample mosaic
viz_classes = ['airplane', 'chair', 'car', 'guitar', 'lamp', 'piano', 'toilet', 'cone']
viz_idx     = [CLASS2IDX[c] for c in viz_classes]
samples_3d  = {}
for pts, lbl in test_ds:
    if lbl in viz_idx and lbl not in samples_3d:
        samples_3d[lbl] = pts.numpy()
    if len(samples_3d) == len(viz_idx):
        break

fig = plt.figure(figsize=(20, 8))
for i, ci in enumerate(viz_idx):
    ax  = fig.add_subplot(2, 4, i+1, projection='3d')
    pts = samples_3d[ci]
    ax.scatter(pts[:,0], pts[:,2], pts[:,1], s=1, c=pts[:,1], cmap='viridis', alpha=.7)
    ax.set_title(IDX2CLASS[ci], fontsize=10); ax.set_axis_off()
plt.suptitle('ModelNet40 — Sample Point Clouds', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig1b_sample_mosaic.png', dpi=150); plt.show()


In [ ]:
# §1 — EDA: mesh vertex count sanity check (first 200 test objects)
pt_counts = []
for path, _ in test_ds.samples[:200]:
    with open(path) as f:
        header = f.readline().strip()
        line = f.readline().strip() if header == 'OFF' else header[3:].strip()
    pt_counts.append(int(line.split()[0]))   # number of vertices

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pt_counts, bins=20, color='steelblue', edgecolor='black', alpha=.8)
ax.axvline(CONFIG['num_points'], color='red', ls='--', label=f"Sample size ({CONFIG['num_points']})")
ax.set_xlabel('Vertices per mesh'); ax.set_ylabel('Count')
ax.set_title('Mesh Vertex Count Distribution (test subset)'); ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig1c_points_histogram.png', dpi=150); plt.show()
print(f"Min {min(pt_counts)}  Max {max(pt_counts)}  Mean {np.mean(pt_counts):.0f}")

In [ ]:
# §2 — Model 1: SortedMLP (naive baseline)
class SortedMLP(nn.Module):
    def __init__(self, num_points=1024, num_classes=40):
        super().__init__()
        self.num_points = num_points
        self.net = nn.Sequential(
            nn.Linear(num_points * 3, 512), nn.ReLU(), nn.Dropout(.3),
            nn.Linear(512, 256),            nn.ReLU(), nn.Dropout(.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        B, N, _ = x.shape
        out = []
        for b in range(B):
            keys = x[b,:,0]*1e8 + x[b,:,1]*1e4 + x[b,:,2]
            out.append(x[b][keys.argsort()].reshape(-1))
        return self.net(torch.stack(out, 0))


In [ ]:
# §2 — Model 2: PointNetVanilla (shared MLP + max-pool, no T-Nets)
class PointNetVanilla(nn.Module):
    def __init__(self, num_classes=40):
        super().__init__()
        def blk(ci, co): return nn.Sequential(nn.Conv1d(ci,co,1), nn.BatchNorm1d(co), nn.ReLU())
        self.enc = nn.Sequential(blk(3,64), blk(64,64), blk(64,64), blk(64,128), blk(128,1024))
        self.cls = nn.Sequential(
            nn.Linear(1024,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.cls(self.enc(x.transpose(1,2)).max(2)[0])


In [ ]:
# §2 — Model 3: Full PointNet (input + feature T-Nets)
class TNet(nn.Module):
    def __init__(self, k=3):
        super().__init__()
        self.k = k
        def blk(ci,co): return nn.Sequential(nn.Conv1d(ci,co,1), nn.BatchNorm1d(co), nn.ReLU())
        self.conv = nn.Sequential(blk(k,64), blk(64,128), blk(128,1024))
        self.fc   = nn.Sequential(
            nn.Linear(1024,512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.out = nn.Linear(256, k*k)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)

    def forward(self, x):
        B = x.size(0)
        h = self.conv(x).max(2)[0]
        return self.out(self.fc(h)).view(B, self.k, self.k) +                torch.eye(self.k, device=x.device).unsqueeze(0)


class PointNet(nn.Module):
    def __init__(self, num_classes=40):
        super().__init__()
        def blk(ci,co): return nn.Sequential(nn.Conv1d(ci,co,1), nn.BatchNorm1d(co), nn.ReLU())
        self.t3   = TNet(3)
        self.e1   = nn.Sequential(blk(3,64),  blk(64,64))
        self.t64  = TNet(64)
        self.e2   = nn.Sequential(blk(64,64), blk(64,128), blk(128,1024))
        self.cls  = nn.Sequential(
            nn.Linear(1024,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(.3),
            nn.Linear(256, num_classes),
        )

    @staticmethod
    def feat_reg(A):
        B, K, _ = A.shape
        I = torch.eye(K, device=A.device).unsqueeze(0)
        diff = torch.bmm(A, A.transpose(1,2)) - I
        return torch.mean(torch.norm(diff, dim=(1,2)))

    def forward(self, x, return_feat=False):
        x = x.transpose(1,2)
        x = torch.bmm(self.t3(x), x)
        x = self.e1(x)
        A = self.t64(x)
        x = torch.bmm(A, x)
        x = self.e2(x)
        feat = x.max(2)[0]
        logits = self.cls(feat)
        if return_feat:
            return logits, feat, A
        return logits


In [ ]:
# §2 — Architecture summary (param counts)
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

_models_info = [
    ('SortedMLP',       SortedMLP(CONFIG['num_points'], CONFIG['num_classes'])),
    ('PointNetVanilla', PointNetVanilla(CONFIG['num_classes'])),
    ('PointNet',        PointNet(CONFIG['num_classes'])),
]
rows = [{'Model': n, 'Params': f'{count_params(m):,}',
         'Params(M)': f'{count_params(m)/1e6:.2f}'} for n, m in _models_info]
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
# §3 — Shared training loop
from tqdm import tqdm

def train_model(model, name, epochs):
    model = model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'],
                             weight_decay=CONFIG['weight_decay'])
    sched = torch.optim.lr_scheduler.StepLR(opt, CONFIG['lr_step'], CONFIG['lr_gamma'])
    crit  = nn.CrossEntropyLoss()
    is_pn = isinstance(model, PointNet)

    hist = {k: [] for k in ['train_loss','train_acc','val_loss','val_acc','lr','epoch_time']}
    best_acc, ckpt = 0., CKPT_DIR / f'{name}_best.pth'

    print(f"\n{'='*60}")
    print(f"  Training {name}  ({epochs} epochs)")
    print(f"{'='*60}")

    for ep in range(1, epochs+1):
        t0 = time.time()

        # ── train ──
        model.train()
        tl, tc, tn = 0., 0, 0
        bar = tqdm(train_loader, desc=f"[{name}] ep {ep:3d}/{epochs} train",
                   leave=False, ncols=90, unit='batch')
        for pts, lbl in bar:
            pts, lbl = pts.to(DEVICE), lbl.to(DEVICE)
            opt.zero_grad()
            if is_pn:
                logits, _, A = model(pts, return_feat=True)
                loss = crit(logits, lbl) + CONFIG['feat_reg_weight'] * PointNet.feat_reg(A)
            else:
                logits = model(pts); loss = crit(logits, lbl)
            loss.backward(); opt.step()
            tl += loss.item()*pts.size(0)
            tc += (logits.argmax(1)==lbl).sum().item()
            tn += pts.size(0)
            bar.set_postfix(loss=f'{tl/tn:.4f}', acc=f'{tc/tn:.3f}')
        bar.close()

        # ── val ──
        model.eval(); vl, vc, vn = 0., 0, 0
        with torch.no_grad():
            for pts, lbl in tqdm(test_loader, desc=f"[{name}] ep {ep:3d}/{epochs}   val",
                                 leave=False, ncols=90, unit='batch'):
                pts, lbl = pts.to(DEVICE), lbl.to(DEVICE)
                lg = model(pts)
                if isinstance(lg, tuple): lg = lg[0]
                vl += crit(lg, lbl).item()*pts.size(0)
                vc += (lg.argmax(1)==lbl).sum().item(); vn += pts.size(0)

        ta, va   = tc/tn, vc/vn
        tl_, vl_ = tl/tn, vl/vn
        et = time.time()-t0
        for k, v in zip(['train_loss','train_acc','val_loss','val_acc','lr','epoch_time'],
                        [tl_, ta, vl_, va, opt.param_groups[0]['lr'], et]):
            hist[k].append(v)

        ckpt_flag = ''
        if va > best_acc:
            best_acc = va
            torch.save(model.state_dict(), ckpt)
            ckpt_flag = '  ← best'

        sched.step()
        print(f"[{name}] ep {ep:3d}/{epochs}  "
              f"train {ta:.4f} / loss {tl_:.4f}  "
              f"val {va:.4f} / loss {vl_:.4f}  "
              f"{et:.1f}s{ckpt_flag}")

    with open(RESULTS_DIR / f'{name}_history.pkl', 'wb') as f:
        pickle.dump(hist, f)
    print(f"\n[{name}] done — best val acc = {best_acc:.4f}  ckpt → {ckpt}\n")
    return hist

In [ ]:
# §3 — Train SortedMLP (30 epochs)
hist_smlp = train_model(SortedMLP(CONFIG['num_points'], CONFIG['num_classes']), 'SortedMLP', 30)


In [ ]:
# §3 — Train PointNetVanilla (50 epochs)
hist_pnv = train_model(PointNetVanilla(CONFIG['num_classes']), 'PointNetVanilla', 50)


In [ ]:
# §3 — Train PointNet (50 epochs, with feature-transform regularisation)
hist_pn = train_model(PointNet(CONFIG['num_classes']), 'PointNet', 50)


In [ ]:
# §4 — Load best checkpoints & collect test-set predictions
def load_model(model, name):
    model.load_state_dict(torch.load(CKPT_DIR / f'{name}_best.pth', map_location=DEVICE))
    return model.to(DEVICE).eval()

all_models = {
    'SortedMLP':       load_model(SortedMLP(CONFIG['num_points'], CONFIG['num_classes']), 'SortedMLP'),
    'PointNetVanilla': load_model(PointNetVanilla(CONFIG['num_classes']), 'PointNetVanilla'),
    'PointNet':        load_model(PointNet(CONFIG['num_classes']),        'PointNet'),
}
histories = {}
for name in all_models:
    with open(RESULTS_DIR / f'{name}_history.pkl', 'rb') as f:
        histories[name] = pickle.load(f)

def get_preds(model):
    ps, ls = [], []
    with torch.no_grad():
        for pts, lbl in test_loader:
            out = model(pts.to(DEVICE))
            lg  = out[0] if isinstance(out, tuple) else out
            ps.extend(lg.argmax(1).cpu().numpy()); ls.extend(lbl.numpy())
    return np.array(ps), np.array(ls)

all_preds = {}
for name, m in all_models.items():
    all_preds[name], gt_labels = get_preds(m)
    print(f"{name}: {(all_preds[name]==gt_labels).mean():.4f}")


In [ ]:
# §4 — Accuracy table (overall + mean-per-class)
def per_class_acc(preds, labels, nc=40):
    return np.array([(preds[labels==c]==c).mean() if (labels==c).any() else 0.
                     for c in range(nc)])

rows = []
for name in all_models:
    p    = all_preds[name]
    oa   = accuracy_score(gt_labels, p)
    mpca = per_class_acc(p, gt_labels).mean()
    h    = histories[name]
    rows.append({'Model': name,
                 'Overall Acc': f'{oa:.4f}',
                 'Mean/Class':  f'{mpca:.4f}',
                 'Params(M)':   f'{count_params(all_models[name])/1e6:.2f}',
                 'Train h':     f'{sum(h["epoch_time"])/3600:.2f}'})

acc_df = pd.DataFrame(rows)
print(acc_df.to_string(index=False))
acc_df.to_csv(RESULTS_DIR / 'table_accuracy.csv', index=False)


In [ ]:
# §4 — Training curves
COLORS = {'SortedMLP':'#e41a1c', 'PointNetVanilla':'#377eb8', 'PointNet':'#4daf4a'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, h in histories.items():
    c  = COLORS[name]
    ep = range(1, len(h['train_acc'])+1)
    axes[0].plot(ep, h['train_acc'],  c=c, ls='-',  lw=1.5, label=name)
    axes[0].plot(ep, h['val_acc'],    c=c, ls='--', lw=1.5, alpha=.7)
    axes[1].plot(ep, h['train_loss'], c=c, ls='-',  lw=1.5, label=name)
    axes[1].plot(ep, h['val_loss'],   c=c, ls='--', lw=1.5, alpha=.7)

for ax, lbl in zip(axes, ['Accuracy', 'Loss']):
    ax.set_xlabel('Epoch'); ax.set_ylabel(lbl)
    ax.set_title(f'Train (—) / Val (- -) {lbl}'); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig3_training_curves.png', dpi=150); plt.show()


In [ ]:
# §4 — Confusion matrices (normalised, 1×3 grid)
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
tick_locs = list(range(0, 40, 5))
for ax, (name, p) in zip(axes, all_preds.items()):
    cm = confusion_matrix(gt_labels, p, normalize='true')
    im = ax.imshow(cm, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_xticks(tick_locs); ax.set_yticks(tick_locs)
    ax.set_xticklabels([IDX2CLASS[i] for i in tick_locs], rotation=90, fontsize=7)
    ax.set_yticklabels([IDX2CLASS[i] for i in tick_locs], fontsize=7)
    plt.colorbar(im, ax=ax, fraction=.046, pad=.04)
plt.suptitle('Normalised Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig4_confusion_matrices.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# §4 — Per-class accuracy delta (PointNet vs SortedMLP, PointNet vs PointNetVanilla)
pca = {name: per_class_acc(all_preds[name], gt_labels) for name in all_models}
delta_pn_naive   = pca['PointNet'] - pca['SortedMLP']
delta_pn_vanilla = pca['PointNet'] - pca['PointNetVanilla']
s_idx = np.argsort(delta_pn_naive)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, delta, title in [
    (axes[0], delta_pn_naive[s_idx],   'PointNet vs SortedMLP (Δ acc)'),
    (axes[1], delta_pn_vanilla[s_idx], 'PointNet vs PointNetVanilla (Δ acc)')]:
    clrs = ['#d73027' if v < 0 else '#4575b4' for v in delta]
    ax.barh([IDX2CLASS[i] for i in s_idx], delta, color=clrs, alpha=.85)
    ax.axvline(0, color='k', lw=.8); ax.set_xlabel('Δ Accuracy')
    ax.set_title(title, fontsize=10); ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig4b_per_class_delta.png', dpi=150, bbox_inches='tight'); plt.show()

# cost/benefit table
cb_rows = [{'Model': n,
            'Params(M)':   f'{count_params(all_models[n])/1e6:.2f}',
            'Best Val Acc':f'{max(histories[n]["val_acc"]):.4f}',
            'Train h':     f'{sum(histories[n]["epoch_time"])/3600:.2f}',
            'Epochs':       len(histories[n]["train_acc"])}
           for n in all_models]
print(pd.DataFrame(cb_rows).to_string(index=False))


In [ ]:
# §5 — Permutation-invariance probe
def eval_order(model, ordering='original', n_trials=1):
    accs = []
    for t in range(n_trials):
        rng = np.random.RandomState(SEED + t)
        correct = total = 0
        with torch.no_grad():
            for pts, lbl in DataLoader(test_ds, 32, shuffle=False,
                                        num_workers=CONFIG['num_workers']):
                if ordering == 'shuffle':
                    for b in range(pts.size(0)):
                        pts[b] = pts[b][torch.from_numpy(rng.permutation(pts.size(1)))]
                elif ordering == 'lex':
                    for b in range(pts.size(0)):
                        k = pts[b,:,0]*1e8 + pts[b,:,1]*1e4 + pts[b,:,2]
                        pts[b] = pts[b][k.argsort()]
                lg = model(pts.to(DEVICE))
                if isinstance(lg, tuple): lg = lg[0]
                correct += (lg.argmax(1).cpu()==lbl).sum().item()
                total   += lbl.size(0)
        accs.append(correct/total)
    return accs

N_SHUF = 10
perm_res = {}
for name, m in all_models.items():
    orig = eval_order(m, 'original')[0]
    shuf = eval_order(m, 'shuffle', N_SHUF)
    lex  = eval_order(m, 'lex')[0]
    perm_res[name] = {'orig': orig, 'shuf_mean': np.mean(shuf),
                      'shuf_std': np.std(shuf), 'lex': lex}
    print(f"{name}  orig={orig:.4f}  shuf={np.mean(shuf):.4f}±{np.std(shuf):.4f}  lex={lex:.4f}")


In [ ]:
# §5 — Plot permutation invariance
fig, ax = plt.subplots(figsize=(10, 5))
names = list(perm_res.keys()); x = np.arange(len(names)); w = .25
ax.bar(x-w, [perm_res[n]['orig']      for n in names], w, label='Original', color='#1f77b4', alpha=.85)
ax.bar(x,   [perm_res[n]['shuf_mean'] for n in names], w,
       label=f'Shuffle (×{N_SHUF})', color='#ff7f0e', alpha=.85,
       yerr=[perm_res[n]['shuf_std'] for n in names], capsize=4)
ax.bar(x+w, [perm_res[n]['lex']       for n in names], w, label='Lex-sort', color='#2ca02c', alpha=.85)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel('Test Accuracy'); ax.set_ylim(0, 1)
ax.set_title('Permutation-Invariance Probe'); ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig5_permutation_invariance.png', dpi=150); plt.show()


In [ ]:
# §6 — Robustness: Gaussian noise
NOISE_SIGMAS  = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2]
N_NOISE_SEEDS = 3

def eval_noise(model, sigma, n_seeds=3):
    accs = []
    for seed in range(n_seeds):
        rng = np.random.RandomState(seed+100)
        c = t = 0
        with torch.no_grad():
            for pts, lbl in DataLoader(test_ds, 32, shuffle=False,
                                        num_workers=CONFIG['num_workers']):
                if sigma > 0:
                    pts = pts + torch.from_numpy(rng.normal(0, sigma, pts.shape).astype(np.float32))
                lg = model(pts.to(DEVICE))
                if isinstance(lg, tuple): lg = lg[0]
                c += (lg.argmax(1).cpu()==lbl).sum().item(); t += lbl.size(0)
        accs.append(c/t)
    return accs

noise_res = {name: [] for name in all_models}
for name, m in all_models.items():
    for s in NOISE_SIGMAS:
        accs = eval_noise(m, s, N_NOISE_SEEDS)
        noise_res[name].append({'s': s, 'mean': np.mean(accs), 'std': np.std(accs)})
    print(name, [(f"{r['s']:.2f}", f"{r['mean']:.4f}") for r in noise_res[name]])


In [ ]:
# §6 — Plot noise robustness
MARKERS = {'SortedMLP':'o', 'PointNetVanilla':'s', 'PointNet':'^'}
fig, ax = plt.subplots(figsize=(10, 6))
for name, res in noise_res.items():
    c  = COLORS[name]
    xs = [r['s']    for r in res]
    ys = [r['mean'] for r in res]
    es = [r['std']  for r in res]
    ax.plot(xs, ys, c=c, marker=MARKERS[name], lw=2, label=name)
    ax.fill_between(xs, [y-e for y,e in zip(ys,es)], [y+e for y,e in zip(ys,es)], color=c, alpha=.15)
ax.axvline(CONFIG['jitter_sigma'], color='gray', ls=':', lw=1.5, label='Train jitter σ')
ax.set_xlabel('Gaussian noise σ'); ax.set_ylabel('Test Accuracy')
ax.set_title('Robustness to Gaussian Noise'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig6a_noise_robustness.png', dpi=150); plt.show()


In [ ]:
# §6 — Robustness: point density
DENSITY_LEVELS = [64, 128, 256, 512, 1024, 2048]

def eval_density(model, model_name, n_pts):
    ds = PointCloudDataset(DATASET_DIR, 'test', n_pts, augment=False, classes=classes)
    ps, ls = [], []
    with torch.no_grad():
        for pts, lbl in DataLoader(ds, 32, shuffle=False, num_workers=CONFIG['num_workers']):
            if model_name == 'SortedMLP':
                B, N, C = pts.shape; tgt = CONFIG['num_points']
                if N < tgt:
                    pts = torch.cat([pts, torch.zeros(B, tgt-N, C)], 1)
                else:
                    pts = pts[:, :tgt]
            lg = model(pts.to(DEVICE))
            if isinstance(lg, tuple): lg = lg[0]
            ps.extend(lg.argmax(1).cpu().numpy()); ls.extend(lbl.numpy())
    return np.array(ps), np.array(ls)

density_res = {}
for name, m in all_models.items():
    density_res[name] = {}
    for n in DENSITY_LEVELS:
        p, l = eval_density(m, name, n)
        density_res[name][n] = {'preds': p, 'labels': l, 'acc': (p==l).mean()}
        print(f"{name} N={n}: {(p==l).mean():.4f}")


In [ ]:
# §6 — Per-class density heatmap + aggregate line
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax, name in zip(axes, all_models):
    mat = np.array([per_class_acc(density_res[name][n]['preds'],
                                  density_res[name][n]['labels'])
                    for n in DENSITY_LEVELS])
    im = ax.imshow(mat, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(40))
    ax.set_xticklabels([IDX2CLASS[i] for i in range(40)], rotation=90, fontsize=6)
    ax.set_yticks(range(len(DENSITY_LEVELS)))
    ax.set_yticklabels([str(n) for n in DENSITY_LEVELS])
    ax.set_ylabel('# Points'); ax.set_title(f'{name} — Per-class acc vs density')
    plt.colorbar(im, ax=ax, fraction=.046, pad=.04)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig6b_density_heatmap.png', dpi=150, bbox_inches='tight'); plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for name in all_models:
    ax.plot(DENSITY_LEVELS, [density_res[name][n]['acc'] for n in DENSITY_LEVELS],
            marker='o', c=COLORS[name], label=name)
ax.set_xscale('log'); ax.set_xlabel('# Points'); ax.set_ylabel('Test Accuracy')
ax.set_title('Accuracy vs Point Density'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig6c_density_line.png', dpi=150); plt.show()


In [ ]:
# §6 — Robustness: spherical point drop — pre-compute occluded clouds
DROP_RADII = [0.0, 0.1, 0.2, 0.3]

def sphere_drop(pts, r, rng):
    if r == 0: return pts
    c    = pts[rng.randint(len(pts))]
    keep = np.linalg.norm(pts - c, axis=1) > r
    rem  = pts[keep]
    return rem if len(rem) >= 64 else pts

rng_drop       = np.random.RandomState(42)
all_pts_raw    = [pts.numpy() for pts, _ in test_ds]
gt_labels_drop = np.array([lbl for _, lbl in test_ds.samples])

dropped_clouds = {}
for r in DROP_RADII:
    dropped_clouds[r] = [sphere_drop(p, r, rng_drop) for p in all_pts_raw]
    print(f"r={r}: mean remaining pts = {np.mean([len(d) for d in dropped_clouds[r]]):.0f}")


In [ ]:
# §6 — Point drop evaluation + violin plot
def eval_drop(model, dropped_list, n_pts=1024):
    rng = np.random.RandomState(42)
    accs, batch_pts, batch_lbl = [], [], []

    def run_batch(bpts, blbl):
        t = torch.tensor(np.stack(bpts), dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            lg = model(t)
            if isinstance(lg, tuple): lg = lg[0]
        return (lg.argmax(1).cpu().numpy() == np.array(blbl)).tolist()

    for i, (d, lbl) in enumerate(zip(dropped_list, gt_labels_drop)):
        idx = rng.choice(len(d), n_pts, replace=(len(d)<n_pts))
        batch_pts.append(d[idx]); batch_lbl.append(lbl)
        if len(batch_pts) == 64 or i == len(dropped_list)-1:
            accs.extend(run_batch(batch_pts, batch_lbl))
            batch_pts, batch_lbl = [], []
    return np.array(accs)

drop_res = {name: {} for name in all_models}
for name, m in all_models.items():
    for r in DROP_RADII:
        drop_res[name][r] = eval_drop(m, dropped_clouds[r])
        print(f"{name} r={r}: mean {drop_res[name][r].mean():.4f}")

import matplotlib.patches as mpatches
palette = ['#e41a1c', '#377eb8', '#4daf4a']
n_m, n_r = len(all_models), len(DROP_RADII)
fig, ax = plt.subplots(figsize=(12, 6))
base = np.arange(n_r) * (n_m + 1.5)
for mi, (name, color) in enumerate(zip(all_models, palette)):
    pos   = base + mi
    data  = [drop_res[name][r] for r in DROP_RADII]
    parts = ax.violinplot(data, positions=pos, widths=.8, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor(color); pc.set_alpha(.6)
    for k in ['cmedians','cmins','cmaxes','cbars']:
        if k in parts: parts[k].set_color(color)
ax.set_xticks(base + (n_m-1)/2)
ax.set_xticklabels([f'r={r}' for r in DROP_RADII])
ax.set_ylabel('Per-sample accuracy')
ax.set_title('Accuracy Distribution Under Spherical Point Drop')
ax.legend(handles=[mpatches.Patch(color=c, label=n) for n,c in zip(all_models, palette)])
ax.set_ylim(-0.05, 1.05); plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig7_drop_violin.png', dpi=150); plt.show()


In [ ]:
# §7 — Critical point sets (PointNet)
def critical_mask(model, pts_t):
    model.eval()
    x = pts_t.to(DEVICE).transpose(1,2)
    with torch.no_grad():
        x = torch.bmm(model.t3(x), x)
        x = model.e1(x)
        A = model.t64(x)
        x = torch.bmm(A, x)
        x = model.e2(x)
    _, idx = x.max(2)
    mask = torch.zeros(pts_t.size(1), dtype=torch.bool)
    mask[torch.unique(idx.cpu())] = True
    return mask.numpy()

pn_model   = all_models['PointNet']
viz_cls    = ['airplane','chair','car','guitar','lamp','piano','toilet','cone']
viz_ci     = [CLASS2IDX[c] for c in viz_cls]
crit_samps = {}
for pts, lbl in test_ds:
    if lbl in viz_ci and lbl not in crit_samps:
        crit_samps[lbl] = pts
    if len(crit_samps) == len(viz_ci): break

fig = plt.figure(figsize=(20, 8))
for i, ci in enumerate(viz_ci):
    pts_t = crit_samps[ci].unsqueeze(0)
    cm    = critical_mask(pn_model, pts_t)
    pts_n = pts_t[0].numpy()
    ax = fig.add_subplot(2, 4, i+1, projection='3d')
    ax.scatter(pts_n[~cm,0], pts_n[~cm,2], pts_n[~cm,1], s=1, c='lightgray', alpha=.4)
    ax.scatter(pts_n[ cm,0], pts_n[ cm,2], pts_n[ cm,1], s=8, c='red',       alpha=.9)
    ax.set_title(f'{IDX2CLASS[ci]}\n({cm.sum()} critical)', fontsize=9)
    ax.set_axis_off()
plt.suptitle('PointNet — Critical Point Sets (red)', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig8_critical_points.png', dpi=150); plt.show()


In [ ]:
# §7 — UMAP of PointNet global features
def extract_feats(model):
    feats, labs = [], []
    with torch.no_grad():
        for pts, lbl in test_loader:
            _, f, _ = model(pts.to(DEVICE), return_feat=True)
            feats.append(f.cpu().numpy()); labs.extend(lbl.numpy())
    return np.vstack(feats), np.array(labs)

print("Extracting features...")
f_pn, l_pn = extract_feats(all_models['PointNet'])
print(f"Features: {f_pn.shape}")
print("Running UMAP...")
emb_pn = umap.UMAP(n_components=2, random_state=SEED).fit_transform(f_pn)

sel_cls = ['airplane','chair','car','guitar','lamp','piano','toilet','cone','sofa','table']
sel_idx = [CLASS2IDX[c] for c in sel_cls]
cmap10  = plt.cm.get_cmap('tab10', 10)

fig, ax = plt.subplots(figsize=(10, 8))
for ci, cls_i in enumerate(sel_idx):
    mask = l_pn == cls_i
    ax.scatter(emb_pn[mask,0], emb_pn[mask,1], s=5, color=cmap10(ci),
               label=IDX2CLASS[cls_i], alpha=.7)
ax.set_title('PointNet — Global Feature Space (UMAP)', fontsize=12)
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.legend(fontsize=8, markerscale=2, ncol=2)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig9_umap.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# §7 — PointNet failure-case gallery
pn_p   = all_preds['PointNet']
fail_i = np.where(pn_p != gt_labels)[0]
print(f"{len(fail_i)} PointNet failures on test set")

# Pick 6 visually interesting failures (wrong class != true class by large margin)
gallery = []
for idx in fail_i:
    pts, lbl = test_ds[idx]
    gallery.append((pts.numpy(), IDX2CLASS[lbl], IDX2CLASS[pn_p[idx]]))
    if len(gallery) == 6: break

fig = plt.figure(figsize=(18, 7))
for i, (pts, true_c, pred_c) in enumerate(gallery):
    ax = fig.add_subplot(2, 3, i+1, projection='3d')
    ax.scatter(pts[:,0], pts[:,2], pts[:,1], s=2, c=pts[:,1], cmap='plasma', alpha=.7)
    ax.set_title(f'True: {true_c}\nPred: {pred_c}', fontsize=9); ax.set_axis_off()
plt.suptitle('PointNet — Failure Cases', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig10_failure_cases.png', dpi=150); plt.show()


In [ ]:
# §8 — LR sweep on PointNet (30 epochs each)
LR_VALUES = [1e-2, 1e-3, 1e-4]
lr_hist   = {}

for lr in LR_VALUES:
    print(f"\n--- lr = {lr:.0e} ---")
    m   = PointNet(CONFIG['num_classes']).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=CONFIG['weight_decay'])
    sch = torch.optim.lr_scheduler.StepLR(opt, CONFIG['lr_step'], CONFIG['lr_gamma'])
    ce  = nn.CrossEntropyLoss()
    va_hist = []

    for ep in range(1, 31):
        m.train()
        for pts, lbl in train_loader:
            pts, lbl = pts.to(DEVICE), lbl.to(DEVICE)
            opt.zero_grad()
            logits, _, A = m(pts, return_feat=True)
            loss = ce(logits, lbl) + CONFIG['feat_reg_weight'] * PointNet.feat_reg(A)
            loss.backward(); opt.step()
        sch.step()

        m.eval(); c = t = 0
        with torch.no_grad():
            for pts, lbl in test_loader:
                pts, lbl = pts.to(DEVICE), lbl.to(DEVICE)
                c += (m(pts).argmax(1)==lbl).sum().item(); t += lbl.size(0)
        va_hist.append(c/t)
        if ep % 5 == 0: print(f"  ep {ep}  val {c/t:.4f}")

    lr_hist[lr] = va_hist


In [ ]:
# §8 — Plot LR sweep
LR_COLORS = {1e-2:'#d62728', 1e-3:'#1f77b4', 1e-4:'#2ca02c'}
fig, ax = plt.subplots(figsize=(10, 5))
for lr, va in lr_hist.items():
    ax.plot(range(1,31), va, c=LR_COLORS[lr], lw=2, label=f'lr={lr:.0e}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val Accuracy')
ax.set_title('PointNet — Learning Rate Sensitivity (30 epochs)')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig11_lr_sweep.png', dpi=150); plt.show()


In [ ]:
# §9 — Results summary
FIGS = [
    'fig1a_class_distribution', 'fig1b_sample_mosaic', 'fig1c_points_histogram',
    'fig3_training_curves', 'fig4_confusion_matrices', 'fig4b_per_class_delta',
    'fig5_permutation_invariance',
    'fig6a_noise_robustness', 'fig6b_density_heatmap', 'fig6c_density_line',
    'fig7_drop_violin', 'fig8_critical_points', 'fig9_umap', 'fig10_failure_cases',
    'fig11_lr_sweep',
]
print("=== Accuracy Summary ===")
print(acc_df.to_string(index=False))
print(f"\nResults: {RESULTS_DIR}")
for f in FIGS:
    p = RESULTS_DIR / f'{f}.png'
    print(f"  {'ok' if p.exists() else 'MISSING':6s}  {f}.png")
